## Анализ данных о Нобелевских лауреатах

### **Контекст проекта**
В этом задании мы будем работать с датасетом о лауреатах Нобелевской премии. Ваша задача — распарсить и проанализировать эти данные, ответив на ключевые вопросы о закономерностях вручения премий, демографических характеристиках лауреатов и исторических тенденциях.

### **Этап 1: Подготовка данных**

**Задача:** На этом этапе вам надо собрать структурированный список словарей с информацией о лауреатах и призах. Приведем начало нужного нам датасета.

``` {code:python}
[{'id': '745',
  'name': 'A. Michael Spence',
  'gender': 'male',
  'birth_year': 1943,
  'country_birth': 'USA',
  'country_now': 'USA',
  'prizes_relevant': [{'prize_amount': 10000000,
    'prize_amount_adjusted': 15547541,
    'award_year': 2001,
    'category_en': 'Economic Sciences',
    'prize_status': 'received'}],
  'type_': 'person'},
 {'id': '102',
  'name': 'Aage N. Bohr',
  'gender': 'male',
  'birth_year': 1922,
  'country_birth': 'Denmark',
  'country_now': 'Denmark',
  'prizes_relevant': [{'prize_amount': 630000,
    'prize_amount_adjusted': 4304697,
    'award_year': 1975,
    'category_en': 'Physics',
    'prize_status': 'received'}],
  'type_': 'person'}]
  ```

Поскольку информация о лауреатах-людях и лауреатах-организациях структурирована по-разному, а также из-за того, что часть полей в json могут отсутстввовать, необходимо реализовать функции для процессинга словарей, которые будут достаточно обобщаемыми для обоих типов данных.

**Что нужно сделать:**

На входе информация о конкретном лауреате представляет из себя кучу вложенных друг в друга словарей. Из этой структуры нам надо получить плоский словарь вида {atribute: value}, где atribute --- это один из небольшого множества интересных нам атрибутов. При этом в исходном файле 
1. пути к атрибутам могут иметь разную длину
2. атрибуты могут отсутствовать
3. атрибуты могут нуждаться в дополнительной обработке (например, поле id в исходных словарях записано как строка.)

Поскольку наивная реализация такого обработчика оказывается довольно громоздкой, вам нужно создать файл json_dict_processing.py, в котором реализован высокоуровневый функционал для решения задачи "обработать вложенный json-словарь"

Главное требование к интерфейсу: 
по (вложенному) словарю о лауреате и конфигу (см ниже) надо получить плоский словарь вида {atribute: value}.

Достичь этого можно, например, таким набором функций

* `extract_nested_value(obj, keys)` - извлекает значение из вложенных структур данных по цепочке ключей

* `process_dictionary_with_config(dictionary, config)` - обрабатывает один словарь согласно словару-конфигу (формат конфига см ниже.)

* `process_list_of_dicts_with_config(list_of_dicts, config)` - обрабатывает список словарей, эта функция нужна чтобы обработать список призов.

* `create_processor(config, list_processor=False)` - создает готовый процессор с предзагруженной конфигурацией. В случае, если list_processor == True, внутрь процессора подается process_list_of_dicts_with_config, иначе process_dictionary_with_config.

**Замечание насчет последней функции.**

Планируемое применение этой функции примерно такое:
`laureate_processor = create_processor(PERSON_CONFIG, list_processor=False)`

То есть, create_processor возвращает функцию, которая реализует функционал process_dictionary_with_config но с уже заполненным конфигом. Это повышает читаемость кода и снижает риск перепутать конфиги из-за дублирования кода.


Помимо этого нужно создать файлы-конфиги для данных по призам, лауреатам-людям, и лауреатам-организациям.

КОНФИГ для функций process_dictionary_with_config и process_list_of_dicts_with_config имеет следующий вид
```python
  {'атрибут_который_берем_сырым': ['путь', 'к', 'данным'],  # или
  'атрибут_который_нужно_обработать': (['путь', 'к', 'данным'], функция_обработки)}
  ```
В случае, если атрибут можно взять в итоговый датасет сырым, в values() словаря-конфига будет список path_list.
Если же атрибут еще надо обработать, для него в поле config[attribute] будет tuple вида (path_list, function).

Примеры атрибутов, которые можно взять сырыми: имя, размер приза. Примеры атрибутов, которые нужно обработать: год вручения приза (в сырых данных есть только полная дата, записанная как строка)

После того, как будет создан core.py, создайте конфиги laureates_configs.py и prizes_configs.py

Постарайтесь провалидировать код с помощью assert-ов, вызываемых в ноутбуке или в .py-файлах.

пример использования: 

assert 1 == True

**Валидация**

В вашем случае assert должен взять простой словарик с одним вложенным полем и какой-нибудь конфиг-файл.
Также убедитесь, что при обработке данных собственно по лауреатам в них нет странностей (смотрите ниже) 

**Атрибуты для ПРИЗА (CONFIG_PRIZE, prizes_configs.py)**

- `prize_amount` - сумма приза → `prizeAmount`
- `prize_amount_adjusted` - скорректированная сумма приза → `prizeAmountAdjusted`
- `award_year` - год вручения приза (преобразуется в целое число) → `awardYear`
- `category_en` - категория на английском → `category.en`
- `prize_status` - статус приза → `prizeStatus`

**Атрибуты для ЧЕЛОВЕКА-ЛАУРЕАТА (CONFIG_PERSON, laureates_configs.py)**

- `id` - уникальный идентификатор → `id`
- `name` - имя лауреата → `knownName.en`
- `gender` - пол → `gender`
- `birth_year` - год рождения (извлекается из даты) → `birth.date`
- `country_birth` - страна рождения → `birth.place.country.en`
- `country_now` - текущая страна → `birth.place.countryNow.en`
- `prizes_relevant` - список призов лауреата → `nobelPrizes` (обрабатывается процессором призов, который вы задаете в конфиге для призов)

**Атрибуты для ОРГАНИЗАЦИИ-ЛАУРЕАТА (CONFIG_ORG, laureates_configs.py)**

- `id` - уникальный идентификатор → `id`
- `name` - название организации → `orgName.en`
- `founded_year` - год основания (извлекается из даты) → `founded.date`
- `country_founded` - страна основания → `founded.place.country.en`
- `country_now` - текущая страна → `founded.place.countryNow.en`
- `prizes_relevant` - список призов организации → `nobelPrizes` (обрабатывается процессором призов, который вы задаете в конфиге для призов)

**Вспомогательные функции:**

- `process_year(year_string)` - извлекает год из строки даты (формат "YYYY-MM-DD"). Функция находится в файле laureates_configs.py
- `prize_processor()` - создает процессор для обработки списков призов. Функция находится в файле prizes_configs.py
- `person_processor()` - создает процессор для данных о людях-лауреатах. Функция находится в файле laureates_configs.py
- `org_processor()` - создает процессор для данных об организациях-лауреатах. Функция находится в файле laureates_configs.py

In [1]:
from core import load_laureates

raw_laureates = load_laureates()


In [2]:
from laureates_configs import process_orgs, process_persons  # реализуйте эти функции согласно описанной выше логике
from dataset_helpers import flatten_prizes
from json_dict_processing import extract_nested_value
from prizes_configs import prize_processor


def get_laureate_data(laureate):
    """
    Определите тип лауреата and соотв образом обработайте данных.
    
    Args:
        laureate: Dictionary with laureate data
        
    Returns:
        Processed laureate data with added 'type_' field ('person' or 'org')
        
    Determines type by:
    - 'knownName' key → person (uses process_persons)
    - 'orgName' key → org (uses process_orgs)
    """
    if 'knownName' in laureate:
        laureate_data = process_persons(laureate)
        laureate_data['type_'] = 'person'
    elif 'orgName' in laureate:
        laureate_data = process_orgs(laureate)
        laureate_data['type_'] = 'org'
    else:
        raise ValueError('Лауреат должен быть человеком или организацией')
    if laureate_data['prizes_relevant'] is None:
        laureate_data['prizes_relevant'] = []
    return laureate_data


laureates_data = [get_laureate_data(laureate) for laureate in raw_laureates]

example = {'category': {'en': 'Physics'}, 'awardYear': '1975'}
assert extract_nested_value(example, ['category', 'en']) == 'Physics'
assert extract_nested_value(example, ['motivation', 'en']) is None

test_prize = prize_processor(example)
assert test_prize['category_en'] == 'Physics'
assert test_prize['award_year'] == 1975
assert prize_processor({})['award_year'] is None

prize_rows = flatten_prizes(laureates_data)

assert all(isinstance(row['id'], int) for row in laureates_data)
assert all(row['type_'] in {'person', 'org'} for row in laureates_data)
assert all(prize['award_year'] is None or isinstance(prize['award_year'], int) for prize in prize_rows)




### **Этап 2: EDA лауреатов**

EDA или Exploratory Data Analysis - первый шаг при работе с любыми данными.

Чтобы начать работать с этой секцией, проведите проведите чисто технический анализ собранных данных.

1. Сколько всего записей?
2. Все ли значения полей заполнены?
3. Если нет, сколько пропусков в каждом из полей? Выведите ответ на этот вопрос в формате `{'field_name': missing_values}`. А какая доля пропусков?
5. Каковы максимальные и минимальные значения поля id и каким годам награждения они соответствуют? Есть ли записи с повторяющимися id? Есть ли пары айди, которые в отсортированном массиве айдишников различаются более, чем на один? 

При реализации ответов на эти вопросы, пользуйтесь теми же принципами, что описаны выше. Можете начать с плохого хакерского кода, однако в таком случае в маркдауне пропишите, как будете его обобщать и разносить по функциям и модулям. (об этом ниже)

В основном EDA пользуйтесь аккуратным кодом, разбитым на функции и файлы.

In [3]:
from dataset_helpers import missing_stats, summarize_ids


print(f'1. Всего записей: {len(laureates_data)}')

counts, shares = missing_stats(laureates_data)
# Если есть хотя бы один пропуск то появляется и ответ на третий пункт
if any(counts.values()):
    print('2. Есть пропуски.')
    print(f'3. Количество пропусков: {counts}')
    print(f'Доля пропусков: {shares}')
else:
    print('2. Пропусков нет.')

first_id_row, last_id_row, duplicate_ids, id_gaps = summarize_ids(laureates_data)

print('\nМинимальный и максимальный ID:')

for row in (first_id_row, last_id_row):
    # Если у лауреата несколько премий, то выводятся все годы
    years = [prize['award_year'] for prize in row['prizes_relevant']]
    print(f'{row["id"]}, год:', *years)

print(f'\nПовторяющиеся ID: {duplicate_ids or "нет"}')
print(f'Разрывы между ID: {id_gaps or "нет"}')


1. Всего записей: 1028
2. Есть пропуски.
3. Количество пропусков: {'id': 0, 'name': 0, 'gender': 0, 'birth_year': 0, 'country_birth': 3, 'country_now': 7, 'prizes_relevant': 0, 'type_': 0, 'founded_year': 1, 'country_founded': 4}
Доля пропусков: {'id': 0.0, 'name': 0.0, 'gender': 0.0, 'birth_year': 0.0, 'country_birth': 0.003, 'country_now': 0.0068, 'prizes_relevant': 0.0, 'type_': 0.0, 'founded_year': 0.0357, 'country_founded': 0.1429}

Минимальный и максимальный ID:
1, год: 1901
1070, год: 2026

Повторяющиеся ID: нет
Разрывы между ID: [(6, 8), (93, 95), (169, 172), (255, 257), (468, 470), (503, 505), (520, 523), (541, 543), (569, 571), (590, 592), (594, 596), (597, 600), (610, 613), (615, 617), (631, 633), (635, 637), (645, 647), (655, 657), (783, 786), (787, 789), (858, 861), (896, 903), (949, 958), (1009, 1011)]



Поскольку это курс по Python, а не по статистике или визуализации данных, мы не задаем жесткие рамки того, что вы должны проанализировать.
Однако это отличная возможность представить себя в роли разработчика базы данных или, например, библиотеки pandas.

В коде нужно соблюдать следующие правила:

1. должны быть реализованы  несколько способов работы с данными (среднее/медианное/максимальное значение метрики, количество примеров, вывод топ-N объектов по значению метрики).
2. анализ выбранных вами тем должен включать в себя анализ метрик в тотале, по категориям и в динамике по годам или десятилетиям.
3. проанализировать стоит 2-3 темы. Вы можете использовать темы ниже или взять свои.
2. вывод ноутбука должен быть понятен обывателю. Это может быть визуализация в matplotlib/seaborn. Сопровождайте вывод markdown-ячейками с вашими  мыслями, пояснениями и инсайтами, которые удалось извлечь.
3. Нельзя использовать pandas и numpy, но можно вдохновляться их интерфейсом и функционалом при дизайне ваших функций.


Из пункта 2 следует, что вам придется работать не только с атрибутами лауреатов, но и самих призов.
Информацию о призах нужно доставать и записывать как атрибут уже лауреата. Функции, работающие с призами (например, факт того, что лауреат отказался от какого-то из призов) можно реализовать в ноутбуке, но если вы создаете более общие вспомогательные функции, выносите их в отдельные модули.

Мы рекомендуем следующую структуру кода:

* `dataset_helpers.py` в котором реализованы фильтрация списка словарей (плоских) по условиям (аналог sql-ного where), а также аналог sql-ного групбая. А также функция для применения агрегаций к списку словарей
* `aggregations.py` в котором реализованы функции расчета среднего/медианы/других эвристик и мб вывод топ n-значений по какой-то метрике.
* `atribute_manager.py` в котором реализован переиспользуемый интерфейс для добавления (и мб удаления) полей.

Замечание насчет group-by. 

На игрушесном примере она может работать примерно так:
```
    Входные данные: [
        {'name': 'Alice', 'country': 'USA', 'age': 25},
        {'name': 'Bob', 'country': 'UK', 'age': 30},
        {'name': 'Charlie', 'country': 'USA', 'age': 25}
    ]
    
    group_by_attributes(data, ['country', 'age']) → {
        ('USA', 25): [
            {'name': 'Alice', 'country': 'USA', 'age': 25},
            {'name': 'Charlie', 'country': 'USA', 'age': 25}
        ],
        ('UK', 30): [
            {'name': 'Bob', 'country': 'UK', 'age': 30}
        ]
    }
```

Замечание насчет agregations.py
1. Агрегации соответственно должны применяться к спискам вида [
    {'name': 'Alice', 'country': 'USA', 'age': 25},
    {'name': 'Charlie', 'country': 'USA', 'age': 25}
]
2. Пользуйтесь модулем statistics из стандартной библиотеки питона.


Эта структура одна из возможных для организации вашего кода, главное, чтобы эти достаточно общие функции блыи переиспользованы

----------

При анализе данных вам понадобятся более конкретные функции, например filter_by_num_prizes или output_metric_by_category. Их пишите прямо в ноутбуке и сопровождайте коментами.

Ниже мы приводим некоторые примеры вопросов, которые может быть интересно задать к данным, но если у вас есть желание, можете поисследовать датасет с любой интересной вам стороны!


**2.1. Топ-статистика по странам**
- Определите топ-5 стран по общему количеству лауреатов
- Постройте аналогичные рейтинги отдельно по категориям (физика, химия, медицина и т.д.)

**2.2. Возраст при награждении**
- Рассчитать средний возраст лауреатов на момент получения **первого** приза. Рассмотрите только лауреатов-людей.
- Исследуйте, как средний возраст получения первого приза различается по категориям
- Различаются ли средний и медианный возрасты? 

**2.3. Гендерное распределение**
- Как менялось количество женщин-лауреатов по десятилетиям?
- Сравните распределение лауреаток по категориям. Выведите значения суммарно за все годы и по декадам.
- В какой из категорий больше всего женщин лауреаток изначально и в последние пару десятилетий?

**2.4. Миграции лауреатов**
- У скольки лауреатов исходная страна и страна на момент вручения нобелевской премии различаются? Всегда ли это различие объясняется миграцией? Добавьте вспомогательное поле 'country_of_origin_adjusted_for_migration', чтобы различие между этим полем и страной на момент вручения объяснялось миграцией.
- Выявите страны с наибольшей долей "оттекших" и "притекших" лауреатов.
- Есть ли оттоки среди организаций?

**2.5. Многократные лауреаты**
- Найдите лауреатов, получившие больше одного приза. Кто получил больше 2 раз? 
- Совпадает ли категория второго приза с первой?  
- Сколько в среднем времени прошло между получением первой и второй премии?

**2.6 Отказы от премий**
- Определите, в каких категориях чаще отказывались от премий?
- Есть ли отказы среди организаций?

### 2.1. Топ-статистика по странам

Топ-5 стран по всему списку лауреатов и отдельно по каждой
категории премии

In [4]:
from dataset_helpers import group_by, top_n_counts, where


def top_countries_for_prizes(prizes):
    # ID ключем считает каждого лауреата один раз
    countries = {prize['laureate_id']: prize['country_now'] for prize in prizes}
    return top_n_counts(countries.values())


def format_ranking(ranking):
    # Страна рядом с количеством лауреатов
    return ', '.join(f'{country} ({count})' for country, count in ranking)


top_countries = top_n_counts(row['country_now'] for row in laureates_data)
print(f'Топ-5 стран: {format_ranking(top_countries)}')

prizes_with_category = where(prize_rows, lambda prize: prize['category_en'] is not None)
prizes_by_category = group_by(prizes_with_category, ['category_en'])

print('\nТоп-5 стран по категориям:')
for (category,), prizes in sorted(prizes_by_category.items()):
    ranking = top_countries_for_prizes(prizes)
    print(f'{category}: {format_ranking(ranking)}')


Топ-5 стран: USA (306), United Kingdom (96), Germany (86), France (66), Japan (32)

Топ-5 стран по категориям:
Chemistry: USA (59), Germany (26), United Kingdom (26), France (12), Japan (9)
Economic Sciences: USA (56), United Kingdom (7), Canada (5), France (5), the Netherlands (4)
Literature: France (12), USA (10), Sweden (7), Poland (7), Germany (7)
Peace: USA (26), France (12), Switzerland (8), United Kingdom (7), Russia (5)
Physics: USA (71), Germany (27), United Kingdom (25), France (13), Japan (12)
Physiology or Medicine: USA (85), United Kingdom (25), Germany (20), France (12), Sweden (8)


#### Вывод

- Больше всего лауреатов в США, Великобритании, Германии, Франции и Японии
- США занимает первое место в пяти категориях по кол-ву лауреато, устпая только Франции в литературе

### 2.2. Возраст при награждении

Для каждого человека ищу первую премию с известным годом и вычитаю год рождения из года награждения, чтобы найти примерный возраст. Сравниваю среднее и медиану для всех лауреатов и по категории первой премии


In [5]:
from aggregations import aggregate_rows


def print_age_stats(label, rows):
    # Общий набор метрик для общего результата и всех категорий
    stats = aggregate_rows(rows, 'age', ['mean', 'median'])
    print(f'{label}: среднее — {stats["mean"]:.1f}, медиана — {stats["median"]:.1f}')


people = where(laureates_data, lambda row: row['type_'] == 'person')
age_rows = []
for person in people:
    birth_year = person['birth_year']
    prizes = where(person['prizes_relevant'], lambda prize: prize['award_year'] is not None)

    if birth_year is not None and prizes:
        # Год и категория одной и той же самой ранней премии
        first_prize = min(prizes, key=lambda prize: prize['award_year'])
        year = first_prize['award_year']
        age_rows.append({
            'age': year - birth_year,
            'category': first_prize['category_en'] or 'отсутствует'
        })

print_age_stats('Возраст в целом', age_rows)
print('\nПо категориям:')
for (category,), rows in sorted(group_by(age_rows, ['category']).items()):
    print_age_stats(category, rows)


Возраст в целом: среднее — 60.6, медиана — 61.0

По категориям:
Chemistry: среднее — 59.4, медиана — 58.0
Economic Sciences: среднее — 67.2, медиана — 67.0
Literature: среднее — 65.0, медиана — 67.0
Peace: среднее — 61.0, медиана — 62.0
Physics: среднее — 57.8, медиана — 56.0
Physiology or Medicine: среднее — 59.0, медиана — 58.0


#### Вывод

- Средний возраст при первом награждении — 60.6 года, а медианный — 61 год. Получилось, что медиана выше
- Самый низкий средний возраст в физике 57,8 года, а самый высокий — в экономических науках 67,2 года


### 2.3. Гендерное распределение

Считаются все женщин-лауреатов, и показывается как менялось их число по десятилетиям и категориям. В пределах каждой группы считаем уникальных
лауреаток

In [6]:
from attribute_manager import add_column


def count_women(prizes):
    # Повторные награждения не считаются
    return len({prize['laureate_id'] for prize in prizes})


def count_women_by_category(prizes):
    # Уникальных лауреаток в каждой категории десятилетия
    return {category or 'Не указана': count_women(rows) for (category,), rows in group_by(prizes, ['category_en']).items()}


women = where(people, lambda row: row.get('gender') == 'female')

# Берем только премии с годом
dated_prizes = where(prize_rows, lambda prize: prize['award_year'] is not None)
dated_prizes = add_column(dated_prizes, 'decade', lambda prize: prize['award_year'] // 10 * 10)
dated_women = where(dated_prizes, lambda prize: prize['gender'] == 'female')

print(f'Всего женщин-лауреатов: {len(women)}')

women_by_decade = group_by(dated_women, ['decade'])
first_decade = min(prize['decade'] for prize in dated_prizes)
last_decade = max(prize['decade'] for prize in dated_prizes)

print('\nПо десятилетиям:')
# Границы берём по всем премиям
for decade in range(first_decade, last_decade + 10, 10):
    prizes = women_by_decade.get((decade,), [])
    counts = count_women_by_category(prizes)
    details = ', '.join(f'{category}: {count}' for category, count in sorted(counts.items()))
    total = count_women(prizes)
    print(f'В {decade}-е всего {total} ({details or "Награждений нет"})')


Всего женщин-лауреатов: 69

По десятилетиям:
В 1900-е всего 3 (Literature: 1, Peace: 1, Physics: 1)
В 1910-е всего 1 (Chemistry: 1)
В 1920-е всего 2 (Literature: 2)
В 1930-е всего 3 (Chemistry: 1, Literature: 1, Peace: 1)
В 1940-е всего 3 (Literature: 1, Peace: 1, Physiology or Medicine: 1)
В 1950-е всего 0 (Награждений нет)
В 1960-е всего 3 (Chemistry: 1, Literature: 1, Physics: 1)
В 1970-е всего 4 (Peace: 3, Physiology or Medicine: 1)
В 1980-е всего 4 (Peace: 1, Physiology or Medicine: 3)
В 1990-е всего 7 (Literature: 3, Peace: 3, Physiology or Medicine: 1)
В 2000-е всего 11 (Chemistry: 1, Economic Sciences: 1, Literature: 3, Peace: 2, Physiology or Medicine: 4)
В 2010-е всего 13 (Chemistry: 1, Economic Sciences: 1, Literature: 3, Peace: 5, Physics: 1, Physiology or Medicine: 2)
В 2020-е всего 16 (Chemistry: 3, Economic Sciences: 1, Literature: 4, Peace: 4, Physics: 2, Physiology or Medicine: 2)


#### Вывод

- Всего 69 женщин-лауреатов
- Можно заметить, что с 1900х по 1940е лауриаток было мало, а в 1950е не было совсем, после чего их кол-во увеличивалось почти каждую декаду
- С 1900х по 1960е чаще всего было по одной лауреатке в каждой категории, а из категорий были литература, физика и исследование мира (если я правильно нагуглил перевод), с 1970х по 1990е состав категорий из менился на литературу и психологию или медицину и максимум лауриаток в категрии уже стало 3, последние десятилетия категорий стало уже 6, а максимум лауреаток 5


---------

## Бонусное задание на +2 балла


### Лингвистический анализ мотиваций**

**5.1. Анализ текстов мотиваций**
- Соберите корпус текстов мотиваций вручения премий. Для этого измените конфиг обработки призов.
- Выделите самые частые топ-50 слов. Какие из этих слов самые частые в принципе в английском языке (такие, как for, the и другие), а какие кажутся важными именно в контексте вручения Нобелевской премии (например, 'chemistry')?
- Самые частые слова, которые также являются частыми во всем английском языке занесите в отдельный конфиг-файл под названием `stopwords.py`. Вычистите их из текстов. Какие слова являются самыми частыми теперь? Сравните множества топ-10 самых частых слов по разным категориям


- Если есть желание, посчитайте TF-IDF + сравнените топ слов по этой метрике с просто самыми популярными словами)

Англоязычные мотивации берем из из поля `motivation_en`, одному тексту на запись лауреат — премия, приведём слова к нижнему регистру и
посчитаем, сколько раз каждое встретилось

In [7]:
from stopwords import STOPWORDS
from text_analysis import top_words


def print_words(title, words):
    # Слово и его частота на отдельной строке
    print(title)
    for word, count in words:
        print(f'{word}: {count}')


with_motivation = where(prize_rows, lambda prize: prize['motivation_en'])
motivations = [prize['motivation_en'] for prize in with_motivation]

print_words('Топ 50 без очистки:', top_words(motivations, 50))


Топ 50 без очистки:
of: 1291
the: 1248
for: 1149
and: 670
in: 449
their: 394
his: 376
to: 278
discovery: 211
discoveries: 172
a: 151
concerning: 124
development: 110
work: 106
on: 101
with: 99
which: 99
by: 82
theory: 74
contributions: 61
has: 60
structure: 58
that: 55
peace: 53
recognition: 48
economic: 45
human: 44
chemical: 43
as: 42
analysis: 42
new: 41
its: 39
synthesis: 38
pioneering: 38
an: 38
fundamental: 37
efforts: 36
between: 34
chemistry: 34
he: 34
her: 32
studies: 31
through: 31
international: 30
method: 30
especially: 29
methods: 29
having: 28
role: 27
research: 26


В исходном рейтинге много служебных слов, например `of`, `the` и `for`. Исключим 50 самых частых слов в английском, сохранённые
в `stopwords.py`, и заново построим топ-50 для мотиваций

In [8]:
print_words('После удаления стоп слов:', top_words(motivations, 50, STOPWORDS))

После удаления стоп слов:
discovery: 211
discoveries: 172
concerning: 124
development: 110
work: 106
theory: 74
contributions: 61
has: 60
structure: 58
peace: 53
recognition: 48
economic: 45
human: 44
chemical: 43
analysis: 42
new: 41
its: 39
synthesis: 38
pioneering: 38
fundamental: 37
efforts: 36
between: 34
chemistry: 34
studies: 31
through: 31
international: 30
method: 30
especially: 29
methods: 29
having: 28
role: 27
research: 26
quantum: 26
cell: 26
nuclear: 26
system: 26
contribution: 26
into: 24
investigations: 24
rights: 24
organic: 23
life: 23
mechanism: 23
high: 23
genetic: 22
reactions: 21
our: 21
light: 20
invention: 20
important: 19


Делим мотивации по категориям премии и для каждой категории 10 самых частых слов после удаления стоп-слов


In [9]:
print('Топ 10 по категориям:')
word_sets = {}
for category, prizes in group_by(with_motivation, ['category_en']).items():
    # возвращается кортеж и при группировке по одному полю
    category_name = category[0]
    texts = [prize['motivation_en'] for prize in prizes]
    words = top_words(texts, 10, STOPWORDS)
    print_words(category_name, words)
    word_sets[category_name] = {word for word, _ in words}

Топ 10 по категориям:
Economic Sciences
theory: 43
economic: 41
analysis: 35
contributions: 18
having: 18
pioneering: 15
research: 13
markets: 10
financial: 8
empirical: 8
Physics
discovery: 100
development: 30
discoveries: 27
theory: 25
contributions: 23
quantum: 22
fundamental: 21
structure: 20
work: 20
physics: 19
Chemistry
development: 52
discovery: 36
chemistry: 33
chemical: 31
synthesis: 28
work: 26
structure: 23
organic: 23
studies: 20
contributions: 17
Literature
has: 30
human: 19
life: 19
poetry: 18
poetic: 17
art: 16
great: 12
power: 11
narrative: 11
recognition: 10
Peace
peace: 53
efforts: 36
work: 32
international: 26
rights: 23
between: 19
war: 19
peaceful: 17
struggle: 16
human: 16
Physiology or Medicine
discoveries: 138
concerning: 97
discovery: 74
cell: 25
system: 25
genetic: 22
cells: 18
development: 16
its: 16
work: 15


Находим слова, которые входят в топ-10 всех шести категорий и слова которые были и в физике и в химии


In [10]:
common_words = set.intersection(*word_sets.values()) if word_sets else set()
print(f'Общие слова всех категорий: {sorted(common_words)}')
shared_words = word_sets['Physics'] & word_sets['Chemistry']
print(f'Общие слова физики и химии: {sorted(shared_words)}')

Общие слова всех категорий: []
Общие слова физики и химии: ['contributions', 'development', 'discovery', 'structure', 'work']


#### Вывод

- До очистки чаще встречаются популярные в англицйском языке `of`, `the` и `for`, из слов, связанных с тематикой премий, в топ 50 вошли `discovery`, `theory` и `chemistry`
- После очистки чаще всего встречаются `discovery`, `discoveries`, `concerning`, `development` и `work`
- Общих слов в топ 10 всех шести категорий нет, а у физики и химии совпадают `contributions`, `development`, `discovery`, `structure` и `work`
